# Day 15 — CTEs: Common Table Expressions
**Estimated time:** 60–75 min
**Datasets:** `sales_orders.csv`, `bw_sales_kpis.csv`

## Learning Objectives
- Write single and chained CTEs to decompose complex query logic
- Build a recursive CTE for org hierarchy traversal
- Refactor deeply nested subqueries into readable CTE chains

In [ ]:
import duckdb
import pandas as pd
import numpy as np
from pathlib import Path

from analytics_bootcamp.config import RAW_DATA_DIR as DATA_DIR
# Load all CSVs
inv   = pd.read_csv(f"{DATA_DIR}/materials_inventory.csv")
sales = pd.read_csv(f"{DATA_DIR}/sales_orders.csv")
cc    = pd.read_csv(f"{DATA_DIR}/cost_center_actuals.csv")
hr    = pd.read_csv(f"{DATA_DIR}/hr_headcount.csv")
bw    = pd.read_csv(f"{DATA_DIR}/bw_sales_kpis.csv")

# Normalize column names
for df in [inv, sales, cc, hr, bw]:
    df.columns = [c.strip().upper() for c in df.columns]

# In-memory DuckDB — register pandas DataFrames as tables
con = duckdb.connect()
con.register("MATERIALS",   inv)
con.register("SALES",       sales)
con.register("COST_CENTER", cc)
con.register("HR",          hr)
con.register("BW_SALES",    bw)

def q(sql):
    return con.sql(sql).df()

# Verify
for tbl, df in [("MATERIALS",inv),("SALES",sales),("COST_CENTER",cc),("HR",hr),("BW_SALES",bw)]:
    n = q(f"SELECT COUNT(*) AS n FROM {tbl}").iloc[0,0]
    print(f"  {tbl:15s}: {n:,} rows")


In [ ]:
# -- Basic CTE: monthly revenue by sales org --
result = q(
    """
    WITH monthly_revenue AS (
        SELECT
            SUBSTR(ERDAT, 1, 7)   AS year_month,
            VKORG,
            SUM(NETWR)            AS revenue,
            COUNT(DISTINCT VBELN) AS orders
        FROM SALES
        WHERE ERDAT IS NOT NULL
        GROUP BY year_month, VKORG
    )
    SELECT *
    FROM monthly_revenue
    ORDER BY year_month DESC, revenue DESC
    LIMIT 20
    """
)
display(result)

In [ ]:
# -- Chained CTEs: revenue -> moving avg -> performance flag --
result = q(
    """
    WITH monthly_revenue AS (
        SELECT SUBSTR(ERDAT, 1, 7) AS year_month, SUM(NETWR) AS revenue
        FROM SALES WHERE ERDAT IS NOT NULL
        GROUP BY year_month
    ),
    revenue_with_avg AS (
        SELECT
            year_month,
            revenue,
            AVG(revenue) OVER (
                ORDER BY year_month
                ROWS BETWEEN 2 PRECEDING AND CURRENT ROW
            ) AS rolling_3m_avg
        FROM monthly_revenue
    ),
    flagged AS (
        SELECT
            year_month,
            revenue,
            ROUND(rolling_3m_avg, 2) AS rolling_3m_avg,
            CASE WHEN revenue < rolling_3m_avg THEN 'Below Average' ELSE 'At or Above' END AS flag
        FROM revenue_with_avg
    )
    SELECT * FROM flagged
    ORDER BY year_month DESC
    LIMIT 24
    """
)
display(result)
print(f"\nMonths below moving avg: {(result['FLAG']=='Below Average').sum()}")

In [ ]:
# -- Recursive CTE: org hierarchy traversal --
# SQLite supports recursive CTEs with RECURSIVE keyword.
# Anchor: root org units. Recursive step: employees under each unit.
result = q(
    """
    WITH RECURSIVE org_tree(node_id, parent_id, label, depth, path) AS (
        SELECT DISTINCT ORGEH, NULL, ORGTX, 0, ORGTX
        FROM HR
        GROUP BY ORGEH
        LIMIT 3
        UNION ALL
        SELECT h.PERNR, h.ORGEH, h.ENAME, ot.depth + 1, ot.path || ' > ' || h.ENAME
        FROM HR h
        INNER JOIN org_tree ot ON h.ORGEH = ot.node_id
        WHERE ot.depth < 2
    )
    SELECT node_id, label, depth, path
    FROM org_tree ORDER BY depth, path
    LIMIT 25
    """
)
display(result)

In [ ]:
# -- Refactoring: nested subquery -> CTE chain --
# BEFORE (nested, hard to read):
before = q(
    """
    SELECT KUNNR, total_revenue FROM (
        SELECT KUNNR, SUM(NETWR) AS total_revenue
        FROM SALES WHERE STATUS != 'CANCELLED' GROUP BY KUNNR
    ) t WHERE total_revenue > (
        SELECT AVG(rev) FROM (
            SELECT SUM(NETWR) AS rev FROM SALES WHERE STATUS != 'CANCELLED' GROUP BY KUNNR
        )
    )
    ORDER BY total_revenue DESC LIMIT 10
    """
)

# AFTER (CTE chain — identical logic, much more readable):
after = q(
    """
    WITH customer_revenue AS (
        SELECT KUNNR, SUM(NETWR) AS total_revenue
        FROM SALES WHERE STATUS != 'CANCELLED'
        GROUP BY KUNNR
    ),
    avg_revenue AS (
        SELECT AVG(total_revenue) AS avg_rev FROM customer_revenue
    )
    SELECT c.KUNNR, c.total_revenue
    FROM customer_revenue c CROSS JOIN avg_revenue a
    WHERE c.total_revenue > a.avg_rev
    ORDER BY c.total_revenue DESC
    LIMIT 10
    """
)
print(f"Row counts match: {len(before) == len(after)}")
display(after)

In [ ]:
# -- CTE: 3-month moving average with below-average flag --
result = q(
    """
    WITH monthly AS (
        SELECT SUBSTR(ERDAT, 1, 7) AS ym, SUM(NETWR) AS revenue
        FROM SALES WHERE ERDAT IS NOT NULL
        GROUP BY ym
    ),
    with_ma AS (
        SELECT
            ym,
            revenue,
            ROUND(AVG(revenue) OVER (ORDER BY ym ROWS BETWEEN 2 PRECEDING AND CURRENT ROW), 2) AS ma3
        FROM monthly
    )
    SELECT
        ym,
        ROUND(revenue, 2)  AS revenue,
        ma3                AS moving_avg_3m,
        CASE WHEN revenue < ma3 THEN 'BELOW AVG' ELSE 'OK' END AS flag
    FROM with_ma
    ORDER BY ym DESC
    LIMIT 20
    """
)
display(result)
print(f"Months below 3M moving avg: {(result['FLAG']=='BELOW AVG').sum()}")

---
## Daily Prompt

**Extend the chained CTE above:**
1. Add a `MOM_CHANGE_PCT` column (month-over-month % change using `LAG`)
2. Flag months where revenue is >20% below the 3-month moving average as `ALERT`
3. Return only the last 18 months

```sql
-- YOUR SOLUTION
WITH monthly AS (
    SELECT SUBSTR(ERDAT, 1, 7) AS ym, SUM(NETWR) AS revenue
    FROM SALES WHERE ERDAT IS NOT NULL GROUP BY ym
),
with_lag AS (
    SELECT
        ym, revenue,
        LAG(revenue) OVER (ORDER BY ym) AS prev_revenue,
        AVG(revenue) OVER (ORDER BY ym ROWS BETWEEN 2 PRECEDING AND CURRENT ROW) AS ma3
    FROM monthly
),
final AS (
    SELECT
        ym, revenue, prev_revenue, ROUND(ma3, 2) AS moving_avg_3m,
        -- YOUR SOLUTION: MOM_CHANGE_PCT
        -- YOUR SOLUTION: ALERT flag (revenue < ma3 * 0.80)
    FROM with_lag
)
SELECT * FROM final ORDER BY ym DESC LIMIT 18
```

> **Hint:** `MOM_CHANGE_PCT = ROUND((revenue - prev_revenue) * 100.0 / NULLIF(prev_revenue, 0), 2)`
> ALERT: `CASE WHEN revenue < ma3 * 0.80 THEN 'ALERT' ELSE 'OK' END`